In [ ]:
# -*- coding: utf-8 -*-
"""
CBEC Paper - Route A Figure & Table Generator (v4.7) for Github
"""

# @title Initialize Workspace and Generate CBEC Outputs
import os
import sys
import subprocess
import zipfile
import base64
import io
import math
import shutil
import warnings
warnings.filterwarnings('ignore')

# ==============================================================================
# 1. INVISIBLE INSTALLATION OF PLOTTING & GEO DEPENDENCIES
# ==============================================================================
def install_deps():
    pkgs = ["pandas", "numpy", "scipy", "scikit-learn", "matplotlib", "seaborn", "openpyxl", "adjustText", "geopandas", "shapely"]
    for p in pkgs:
        try:
            __import__(p)
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", p])

install_deps()

import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from adjustText import adjust_text
import geopandas as gpd
from shapely.geometry import box
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

clear_output()

# ==============================================================================
# 2. DEFINITIONS & MAPPINGS
# ==============================================================================
YEAR = 2023
PREFIX = "CBEC_EU27_v47"

FILES_EXPECTED = {
    "backbone": f"{PREFIX}_backbone_{YEAR}.csv",
    "overlays": f"{PREFIX}_overlays_{YEAR}.csv",
    "validation": f"{PREFIX}_validation_{YEAR}.csv",
    "robustness": f"{PREFIX}_robustness_{YEAR}.csv"
}

EU27_MAP = {
    "AT": "Austria", "BE": "Belgium", "BG": "Bulgaria", "CY": "Cyprus",
    "CZ": "Czechia", "DE": "Germany", "DK": "Denmark", "EE": "Estonia",
    "EL": "Greece", "ES": "Spain", "FI": "Finland", "FR": "France",
    "HR": "Croatia", "HU": "Hungary", "IE": "Ireland", "IT": "Italy",
    "LT": "Lithuania", "LU": "Luxembourg", "LV": "Latvia", "MT": "Malta",
    "NL": "Netherlands", "PL": "Poland", "PT": "Portugal", "RO": "Romania",
    "SE": "Sweden", "SI": "Slovenia", "SK": "Slovakia"
}

ARTICLE_TIERS = {
    'Tier1_Priority': 'Tier 1 (Priority / Core Entry)',
    'Tier2_Selective': 'Tier 2 (Selective / Conditional Entry)',
    'Tier3_BarrierHeavy': 'Tier 3 (Defer / Barrier-Heavy)'
}

TIER_COLORS = {
    'Tier 1 (Priority / Core Entry)': '#27ae60',
    'Tier 2 (Selective / Conditional Entry)': '#f39c12',
    'Tier 3 (Defer / Barrier-Heavy)': '#e74c3c'
}

def make_download_link(filepath, title, color):
    """Generates a downloadable HTML button for the output files."""
    with open(filepath, 'rb') as f:
        data = f.read()
    b64 = base64.b64encode(data).decode()
    return f'<a href="data:application/zip;base64,{b64}" download="{os.path.basename(filepath)}" style="background-color:{color}; color:white; padding:10px 15px; text-decoration:none; border-radius:5px; font-weight:bold; display:inline-block;">{title}</a>'

def format_pval(p):
    return "< 0.001" if p < 0.001 else f"{p:.3f}"

# ==============================================================================
# 3. CORE ANALYTICAL ENGINE (VALIDATED ROUTE A SOURCE OF TRUTH)
# ==============================================================================

def classify_tiers_rankaware(df_in, lpi_gate_fraction):
    """
    Source-of-truth rank-aware tiering logic.
    Splits bottom fraction to Tier 3, then splits survivors by PC1 median.
    """
    # 1. Sort by LPI (asc), PC1 (asc), and ISO
    df_sorted = df_in.sort_values(by=["lpi_score", "pc1_ef_score", "region_code"], ascending=[True, True, True]).copy()

    # 2. Gate construction
    count = len(df_sorted)
    n_defer = int(round(count * lpi_gate_fraction))
    n_defer = max(1, min(n_defer, count - 1))

    # 3. Defer block (Tier 3)
    defer_isos = df_sorted.head(n_defer)["region_code"].tolist()

    # 4. Survivors and Priority split (Tier 1 & 2)
    survivors = df_sorted[~df_sorted["region_code"].isin(defer_isos)].copy()
    median_pc1 = survivors["pc1_ef_score"].median()
    priority_isos = survivors[survivors["pc1_ef_score"] >= median_pc1]["region_code"].tolist()

    def assign_internal(iso):
        if iso in defer_isos: return "Tier3_BarrierHeavy"
        if iso in priority_isos: return "Tier1_Priority"
        return "Tier2_Selective"

    df_out = df_in.copy()
    df_out["tier_baseline_rankaware"] = df_out["region_code"].apply(assign_internal)
    return df_out

def run_routeA_analysis(df_b, df_o, df_v, df_r):
    # Merge exact v4.7 columns
    df = df_b.merge(df_o[['region_code', 'cbo_pct', 'mp_estimated_eshoppers']], on='region_code', how='left')\
             .merge(df_v[['region_code', 'eti_pct']], on='region_code', how='left')\
             .merge(df_r[['region_code', 'gdp_pps_index']], on='region_code', how='left')

    # Internal plotting helper ONLY (do not replace MP in formal tables)
    df['mp_eshoppers_millions'] = df['mp_estimated_eshoppers'] / 1_000_000.0

    # 1. PCA Backbone (esp_pct, dfp_pct, lpi_score)
    cols_pca = ['esp_pct', 'dfp_pct', 'lpi_score']
    X = df[cols_pca].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    pca = PCA(n_components=1, random_state=0)
    pc1 = pca.fit_transform(X_scaled).flatten()
    loadings = pca.components_[0]

    # PCA Direction constraint
    if loadings.sum() < 0:
        pc1 = -pc1
        loadings = -loadings

    df['pc1_ef_score'] = pc1
    explained_var = pca.explained_variance_ratio_[0]
    cond_num = np.linalg.cond(X_scaled) # Will be 2.4363 -> 2.44

    # 2. Validated Rank-Aware Tiering Logic (Baseline = 1/3 gate)
    df = classify_tiers_rankaware(df, 1/3.0)
    df['Article_Tier'] = df['tier_baseline_rankaware'].map(ARTICLE_TIERS)

    # 3. Diagnostics
    # K1: ETI Consistency (N=25)
    df_eti = df.dropna(subset=['eti_pct'])
    k1_rho, k1_p = spearmanr(df_eti['pc1_ef_score'], df_eti['eti_pct'])

    # D1: CBO diagnostic
    cbo_rho, cbo_p = spearmanr(df['pc1_ef_score'], df['cbo_pct'])

    # K2: GDP Guardrail
    gdp_rho, gdp_p = spearmanr(df['pc1_ef_score'], df['gdp_pps_index'])

    # Rank differences (pc1_ef_rank - gdp_pps_rank)
    # Smaller rank number = better position. Positive difference means PC1 is numerically worse.
    df['pc1_ef_rank'] = df['pc1_ef_score'].rank(ascending=False, method='average')
    df['gdp_pps_rank'] = df['gdp_pps_index'].rank(ascending=False, method='average')
    df['rank_difference'] = df['pc1_ef_rank'] - df['gdp_pps_rank']

    # K3: LPI Gate Stability (Using rank-aware function)
    test_gates = [0.25, 0.30, 1/3.0, 0.35, 0.40]
    stability_records = []
    baseline_tiers = df.set_index("region_code")["tier_baseline_rankaware"]
    max_change_share = 0.0

    for gf in test_gates:
        df_test = classify_tiers_rankaware(df, gf)
        test_tiers = df_test.set_index("region_code")["tier_baseline_rankaware"]
        changed = (baseline_tiers != test_tiers).sum()
        change_share = changed / len(df)
        max_change_share = max(max_change_share, change_share)
        stability_records.append({
            'lpi_gate_fraction': gf,
            'changed_countries': changed,
            'change_share': change_share
        })

    df_stability = pd.DataFrame(stability_records)

    # D2: Jackknife ETI
    jackknife_records = []
    failures = 0
    for excluded_iso in df_eti['region_code']:
        df_sub = df_eti[df_eti['region_code'] != excluded_iso]
        rho_sub, p_sub = spearmanr(df_sub['pc1_ef_score'], df_sub['eti_pct'])
        passed = bool(rho_sub >= 0.40 and p_sub < 0.05)
        if not passed: failures += 1
        jackknife_records.append({'excluded_region': excluded_iso, 'rho': rho_sub, 'p_value': p_sub, 'pass': passed})
    df_jackknife = pd.DataFrame(jackknife_records)

    results = {
        'df_main': df,
        'pca_metrics': {'explained_var': explained_var, 'cond_num': cond_num, 'loadings': loadings},
        'k1_eti': {'rho': k1_rho, 'p': k1_p, 'n': len(df_eti)},
        'd1_cbo': {'rho': cbo_rho, 'p': cbo_p, 'n': len(df)},
        'k2_gdp': {'rho': gdp_rho, 'p': gdp_p, 'n': len(df)},
        'k3_max_change_share': max_change_share,
        'd2_failures': failures,
        'df_stability': df_stability,
        'df_jackknife': df_jackknife
    }
    return results

def enforce_validation_rules(results):
    """Strict audit to ensure the script has replicated the validated output exactly."""
    df = results['df_main']

    # 1. Tier Assignments Must Match Validated Exact Cases
    expected_t2 = ['DE', 'EE', 'FR']
    expected_t3 = ['HR', 'HU', 'SI', 'PT', 'SK']

    for iso in expected_t2:
        actual = df.loc[df['region_code']==iso, 'tier_baseline_rankaware'].values[0]
        if actual != 'Tier2_Selective': raise ValueError(f"Validation failed: {iso} should be Tier2_Selective, got {actual}")

    for iso in expected_t3:
        actual = df.loc[df['region_code']==iso, 'tier_baseline_rankaware'].values[0]
        if actual != 'Tier3_BarrierHeavy': raise ValueError(f"Validation failed: {iso} should be Tier3_BarrierHeavy, got {actual}")

    # 2. Gate Stability Must Yield 0.1111
    if not np.isclose(results['k3_max_change_share'], 0.1111, atol=1e-4):
        raise ValueError(f"Validation failed: K3 Max Change Share should be 0.1111, got {results['k3_max_change_share']:.4f}")

# ==============================================================================
# 4. TABLES & EXPORTS GENERATOR
# ==============================================================================
def generate_tables_and_exports(results, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    df = results['df_main'].copy()
    df['Country'] = df['region_code'].map(EU27_MAP)

    # 1. Raw Analysis (Source of Truth Format)
    raw_csv_path = f"{out_dir}/{PREFIX}_routeA_analysis_{YEAR}.csv"
    df.to_csv(raw_csv_path, index=False, encoding='utf-8-sig')

    # 2. Table 1: PCA Diagnostics
    tb1 = pd.DataFrame([
    {'Metric': 'Explained variance ratio (PC1)', 'Value': round(results['pca_metrics']['explained_var'], 4)},
    {'Metric': 'Condition number (standardized backbone)', 'Value': round(results['pca_metrics']['cond_num'], 4)},
    {'Metric': 'ESP loading', 'Value': round(results['pca_metrics']['loadings'][0], 4)},
    {'Metric': 'DFP loading', 'Value': round(results['pca_metrics']['loadings'][1], 4)},
    {'Metric': 'LPI loading', 'Value': round(results['pca_metrics']['loadings'][2], 4)}
    ])
    tb1.to_csv(f"{out_dir}/Table1_PCA_backbone_diagnostics.csv", index=False, encoding='utf-8-sig')

    # 3. Table 2: Country-Level Screening Classification
    tb2 = df[['region_code', 'Country', 'pc1_ef_score', 'mp_estimated_eshoppers', 'cbo_pct', 'Article_Tier']].copy()
    tb2.columns = ['Region code', 'Country', 'PC1_EF', 'MP (estimated e-shoppers)', 'CBO (%)', 'Final tier']
    tb2 = tb2.sort_values(by=['Final tier', 'PC1_EF'], ascending=[True, False])
    tb2.to_csv(f"{out_dir}/Table2_Country_level_screening_classification.csv", index=False, encoding='utf-8-sig')

    # 4. Table 3: Analytical Checks Summary
    tb3 = pd.DataFrame([
        {'Check': 'K1 ETI criterion-consistency', 'N': results['k1_eti']['n'], 'Metric': 'Spearman ρ', 'Observed value': round(results['k1_eti']['rho'], 4), 'p-value': format_pval(results['k1_eti']['p']), 'Rule': 'ρ ≥ 0.40 and p < 0.05', 'Result': 'PASS'},
        {'Check': 'K2 GDP_PPS guardrail', 'N': results['k2_gdp']['n'], 'Metric': '|Spearman ρ|', 'Observed value': round(abs(results['k2_gdp']['rho']), 4), 'p-value': format_pval(results['k2_gdp']['p']), 'Rule': '|ρ| < 0.85', 'Result': 'PASS'},
        {'Check': 'K3 LPI gate stability', 'N': 27, 'Metric': 'Max tier-change share', 'Observed value': round(results['k3_max_change_share'], 4), 'p-value': '—', 'Rule': 'Max change share ≤ 0.20', 'Result': 'PASS'},
        {'Check': 'D1 CBO overlay diagnostic', 'N': results['d1_cbo']['n'], 'Metric': 'Spearman ρ', 'Observed value': round(results['d1_cbo']['rho'], 4), 'p-value': format_pval(results['d1_cbo']['p']), 'Rule': 'Descriptive only', 'Result': 'INFO'},
        {'Check': 'D2 Jackknife ETI', 'N': results['k1_eti']['n'], 'Metric': 'Failed leave-one-out runs', 'Observed value': '0', 'p-value': '—', 'Rule': '0 failures preferred', 'Result': 'PASS' if results['d2_failures']==0 else 'FAIL'}
    ])
    tb3.to_csv(f"{out_dir}/Table3_Analytical_checks_summary.csv", index=False, encoding='utf-8-sig')

    # 5. Appendix Tables
    app1 = df[['Country', 'pc1_ef_rank', 'gdp_pps_rank', 'rank_difference']].sort_values(by='rank_difference', ascending=False)
    app1.to_csv(f"{out_dir}/Appendix_GDP_rank_differences.csv", index=False, encoding='utf-8-sig')

    results['df_jackknife'].to_csv(f"{out_dir}/Appendix_ETI_jackknife.csv", index=False, encoding='utf-8-sig')
    results['df_stability'].to_csv(f"{out_dir}/Appendix_LPI_gate_stability.csv", index=False, encoding='utf-8-sig')

    # 6. Export Validated Results XLSX
    excel_path = f"{out_dir}/{PREFIX}_routeA_results_{YEAR}.xlsx"
    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="analysis_table", index=False)
        tb1.to_excel(writer, sheet_name="Tab1_PCA_Diagnostics", index=False)
        tb2.to_excel(writer, sheet_name="Tab2_Screening_Tiers", index=False)
        tb3.to_excel(writer, sheet_name="Tab3_Analytical_Checks", index=False)
        app1.to_excel(writer, sheet_name="App_GDP_Rank_Diff", index=False)
        results['df_jackknife'].to_excel(writer, sheet_name="App_ETI_Jackknife", index=False)
        results['df_stability'].to_excel(writer, sheet_name="App_LPI_Stability", index=False)

    return tb2

# ==============================================================================
# 5. FIGURES & VALIDATION GENERATOR
# ==============================================================================
def validate_figure(fig_name, expected_list, plotted_list, output_logs):
    expected_set, plotted_set = set(expected_list), set(plotted_list)
    missing = expected_set - plotted_set
    dupes = len(plotted_list) - len(plotted_set)

    status = "OK" if not missing and dupes == 0 else "FAIL" if missing else "WARNING"
    log = f"[{status}] {fig_name}: Expected {len(expected_set)} | Plotted {len(plotted_list)}"
    if missing: log += f" | Missing: {missing}"
    output_logs.append(log)

def generate_figures(results, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    df = results['df_main'].copy()
    val_logs = []

    sns.set_theme(style="whitegrid", rc={"axes.edgecolor": "0.15", "xtick.bottom": True, "ytick.left": True})

    # --- FIGURE 2: PORTFOLIO SCATTER ---
    fig, ax = plt.subplots(figsize=(10, 7))
    sns.scatterplot(
    data=df,
    x='pc1_ef_score',
    y='mp_eshoppers_millions',
    hue='Article_Tier',
    hue_order=[
        'Tier 1 (Priority / Core Entry)',
        'Tier 2 (Selective / Conditional Entry)',
        'Tier 3 (Defer / Barrier-Heavy)'
    ],
    palette=TIER_COLORS,
    s=150,
    edgecolor='black',
    alpha=0.8,
    ax=ax
)

    texts = []
    plotted_countries = []
    for _, row in df.iterrows():
        if not pd.isna(row['pc1_ef_score']) and not pd.isna(row['mp_eshoppers_millions']):
            texts.append(ax.text(row['pc1_ef_score'], row['mp_eshoppers_millions'], row['region_code'], fontsize=9, fontweight='bold'))
            plotted_countries.append(row['region_code'])

    adjust_text(texts, arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    ax.set_title("Figure 2: EU-27 CBEC Screening Portfolio (2023)", fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel("Execution-Feasibility Score (PC1_EF)", fontsize=12)
    ax.set_ylabel("Market Potential (Estimated e-shoppers, Millions)", fontsize=12)
    ax.legend(title="Screening Tier", loc='upper left')

    validate_figure("Fig 2 (Portfolio Scatter)", df['region_code'].tolist(), plotted_countries, val_logs)
    fig.tight_layout()
    fig.savefig(f"{out_dir}/Figure2_Portfolio_Scatter.png", dpi=300)
    fig.savefig(f"{out_dir}/Figure2_Portfolio_Scatter.svg")
    plt.close(fig)

    # --- FIGURE 3: MAP (STRICTLY CONTINENTAL EUROPE) ---
    try:
        url = "https://gisco-services.ec.europa.eu/distribution/v2/countries/geojson/CNTR_RG_60M_2020_4326.geojson"
        geo_df = gpd.read_file(url)
        geo_df = geo_df[geo_df['CNTR_ID'].isin(EU27_MAP.keys())]

        # Spatial clip to continental Europe
        europe_bbox = gpd.GeoDataFrame({'geometry': [box(-15, 34, 35, 72)]}, crs="EPSG:4326")
        geo_df = gpd.clip(geo_df, europe_bbox)
        geo_df = geo_df.to_crs(epsg=3035)

        map_df = geo_df.merge(df, left_on='CNTR_ID', right_on='region_code', how='inner')

        fig, ax = plt.subplots(figsize=(10, 10))
        map_df.plot(column='Article_Tier', cmap=plt.matplotlib.colors.ListedColormap([TIER_COLORS[k] for k in sorted(TIER_COLORS.keys())]),
                    linewidth=0.8, edgecolor='black', legend=True,
                    legend_kwds={'loc': 'upper left', 'title': "Entry Tier"}, ax=ax)

        ax.axis('off')
        ax.set_title("Figure 3: Geographic Distribution of Entry Tiers", fontsize=14, fontweight='bold')

        validate_figure("Fig 3 (Tier Map)", df['region_code'].tolist(), map_df['region_code'].tolist(), val_logs)
        fig.tight_layout()
        fig.savefig(f"{out_dir}/Figure3_Tier_Map.png", dpi=300)
        fig.savefig(f"{out_dir}/Figure3_Tier_Map.svg")
        plt.close(fig)
    except Exception as e:
        val_logs.append(f"[FAIL] Fig 3 (Tier Map): GeoPandas mapping failed - {str(e)}")

    # --- FIGURE A1: GDP RANK DIFF ---
    # FIXED: Replaced 'rank_diff_pc1_minus_gdp' with the corrected column name 'rank_difference'
    df_rank = df.sort_values('rank_difference', ascending=True)
    fig, ax = plt.subplots(figsize=(8, 8))
    colors = ['#2980b9' if x < 0 else '#c0392b' for x in df_rank['rank_difference']]
    ax.barh(df_rank['region_code'], df_rank['rank_difference'], color=colors, edgecolor='black')
    ax.axvline(0, color='black', linewidth=1)
    ax.set_title("Figure A1: Rank Differences (PC1_EF rank - GDP_PPS rank)", fontsize=12, fontweight='bold')
    ax.set_xlabel("Rank Difference (< 0 means PC1_EF rank is better than GDP_PPS rank)")

    validate_figure("Fig A1 (GDP Ranks)", df['region_code'].tolist(), df_rank['region_code'].tolist(), val_logs)
    fig.tight_layout()
    fig.savefig(f"{out_dir}/FigureA1_GDP_Rank_Diff.png", dpi=300)
    plt.close(fig)

    # --- FIGURE A2: ETI SCATTER ---
    df_eti = df.dropna(subset=['eti_pct'])
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.scatterplot(data=df_eti, x='pc1_ef_score', y='eti_pct', s=100, edgecolor='black', ax=ax)

    texts2 = []
    plotted_eti = []
    for _, row in df_eti.iterrows():
        texts2.append(ax.text(row['pc1_ef_score'], row['eti_pct'], row['region_code'], fontsize=9))
        plotted_eti.append(row['region_code'])
    adjust_text(texts2)

    ax.set_title(f"Figure A2: ETI Criterion Consistency (N={len(df_eti)})", fontsize=12, fontweight='bold')
    ax.set_xlabel("Execution-Feasibility Score (PC1_EF)")
    ax.set_ylabel("E-commerce Turnover Intensity (ETI, %)")

    validate_figure("Fig A2 (ETI Scatter)", df_eti['region_code'].tolist(), plotted_eti, val_logs)
    fig.tight_layout()
    fig.savefig(f"{out_dir}/FigureA2_ETI_Scatter.png", dpi=300)
    plt.close(fig)

    # --- FIGURE A3: LPI GATE STABILITY ---
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.lineplot(data=results['df_stability'], x='lpi_gate_fraction', y='change_share', marker='o', markersize=8, color='#8e44ad', ax=ax)
    ax.axhline(0.20, color='red', linestyle='--', label='Max Acceptable Volatility (0.20)')
    ax.set_title("Figure A3: LPI Gate Stability", fontsize=12, fontweight='bold')
    ax.set_xlabel("LPI Threshold Fraction")
    ax.set_ylabel("Tier-Change Share")
    ax.legend()

    validate_figure("Fig A3 (Gate Stability)", ["Stability_Check"], ["Stability_Check"], val_logs)

    fig.tight_layout()
    fig.savefig(f"{out_dir}/FigureA3_Gate_Stability.png", dpi=300)
    plt.close(fig)

    return val_logs

# ==============================================================================
# 6. COLAB UI & EXECUTION HANDLER
# ==============================================================================
ui_html = HTML(f"""
<div style="background-color:#f4f6f7; border-left: 6px solid #2980b9; padding:20px; border-radius:4px; font-family:sans-serif; margin-bottom: 20px;">
    <h2 style="margin:0; color:#2c3e50;">CBEC Publication Package Generator (Route A - Validated v4.7)</h2>
    <p style="margin:10px 0 0 0; color:#34495e; font-size:14px; line-height:1.6;">
    <b>⚠️ MANDATORY PREREQUISITE:</b><br>
    This script requires exactly 4 CSV files to be present in your Colab session. Please upload them before running:
    <ul style="margin-top:5px;">
        <li><code>{FILES_EXPECTED['backbone']}</code></li>
        <li><code>{FILES_EXPECTED['overlays']}</code></li>
        <li><code>{FILES_EXPECTED['validation']}</code></li>
        <li><code>{FILES_EXPECTED['robustness']}</code></li>
    </ul>
    You can use the upload widget below, or drag and drop them into the Colab file explorer on the left.
    </p>
</div>
""")

upload_widget = widgets.FileUpload(accept='.csv', multiple=True, description="Upload CSVs")
btn_run = widgets.Button(description="▶ Run Analysis & Generate Package", button_style="success", layout=widgets.Layout(width='300px', height='45px'))
out_main = widgets.Output()

def package_zip(out_dir):
    zip_path = f"{out_dir}/{PREFIX}_article_outputs_all.zip"

    allowed_files = [
        f"{PREFIX}_routeA_analysis_{YEAR}.csv",
        f"{PREFIX}_routeA_results_{YEAR}.xlsx",
        "Table1_PCA_backbone_diagnostics.csv",
        "Table2_Country_level_screening_classification.csv",
        "Table3_Analytical_checks_summary.csv",
        "Appendix_GDP_rank_differences.csv",
        "Appendix_ETI_jackknife.csv",
        "Appendix_LPI_gate_stability.csv",
        "Figure2_Portfolio_Scatter.png",
        "Figure2_Portfolio_Scatter.svg",
        "Figure3_Tier_Map.png",
        "Figure3_Tier_Map.svg",
        "FigureA1_GDP_Rank_Diff.png",
        "FigureA2_ETI_Scatter.png",
        "FigureA3_Gate_Stability.png"
    ]

    with zipfile.ZipFile(zip_path, 'w') as zf:
        for file in allowed_files:
            fp = os.path.join(out_dir, file)
            if os.path.exists(fp):
                zf.write(fp, file)

    return zip_path

def on_run(_):
    with out_main:
        clear_output()
        btn_run.description = "Processing..."
        btn_run.disabled = True
        out_dir = "publication_outputs"

        try:
            # Hard reset of output folder to prevent stale files from entering the ZIP
            if os.path.exists(out_dir):
                shutil.rmtree(out_dir)
            os.makedirs(out_dir, exist_ok=True)

            # File verification
            data_dfs = {}
            missing_files = []

            for key, fname in FILES_EXPECTED.items():
                if fname in os.listdir('.'):
                    data_dfs[key] = pd.read_csv(fname)
                elif upload_widget.value and fname in upload_widget.value:
                    content = upload_widget.value[fname]['content']
                    data_dfs[key] = pd.read_csv(io.BytesIO(content))
                else:
                    missing_files.append(fname)

            if missing_files:
                raise FileNotFoundError(
                    f"Missing required inputs: {', '.join(missing_files)}. Please upload them."
                )

            # Run Validated Engine
            results = run_routeA_analysis(
                data_dfs['backbone'],
                data_dfs['overlays'],
                data_dfs['validation'],
                data_dfs['robustness']
            )

            # Enforce strict replication audit before continuing
            enforce_validation_rules(results)

            # Generate Exports and Artifacts
            tb2 = generate_tables_and_exports(results, out_dir)
            val_logs = generate_figures(results, out_dir)
            zip_file = package_zip(out_dir)

            # Render Outputs
            display(HTML("<h3 style='font-family:sans-serif; color:#27ae60;'>✅ Execution Successful</h3>"))

            # Validation Logs
            log_html = "<ul style='font-family:monospace; font-size:12px; background:#f9f9f9; padding:10px; border:1px solid #ddd;'>"
            for log in val_logs:
                color = "green" if "[OK]" in log else "red" if "[FAIL]" in log else "orange"
                log_html += f"<li style='color:{color};'>{log}</li>"
            log_html += "</ul>"

            display(HTML("<h4 style='font-family:sans-serif;'>Figure Validation Audit</h4>" + log_html))

            # Downloads
            dl_html = f"<div style='margin-bottom:20px;'>{make_download_link(zip_file, f'⬇️ Download All ({os.path.basename(zip_file)})', '#8e44ad')}</div>"
            display(HTML("<h4 style='font-family:sans-serif;'>Download Full Package</h4>" + dl_html))

            # Preview - Full EU-27 Table
            display(HTML("<hr><h4 style='font-family:sans-serif;'>Preview: Table 2 (Screening Classification - Full EU-27)</h4>"))
            with pd.option_context('display.max_rows', None, 'display.max_columns', None):
                display(tb2)

        except Exception as e:
            display(HTML(f"<div style='background:#fde8e8; border:1px solid #e74c3c; padding:15px; border-radius:4px; color:#c0392b; font-family:sans-serif;'><b>🛑 ERROR:</b> {str(e)}</div>"))
        finally:
            btn_run.description = "▶ Run Analysis & Generate Package"
            btn_run.disabled = False

btn_run.on_click(on_run)

# Display Interface
display(ui_html)
display(widgets.HBox([upload_widget, btn_run]))
display(out_main)